In [1]:
import duckdb
import pandas as pd
from scipy import stats

con = duckdb.connect("../Data/processed/project.duckdb")

trades_with_trader_df = con.execute("SELECT * FROM trades_with_trader WHERE traderId IS NOT NULL").fetchdf()
rebuilt_account_summary = pd.read_csv("../Data/processed/rebuilt_account_summary.csv")


DATA : The data being used is essentially the masterdataset stored as a table in project.duckdb as trades_with traders. 

- We are treating email as a primary key. As in, an emailID means a unique trader. According to PDPA laws, we cannot keep the email, tele ID column and therefore we created a "traderID" column, which is a unique identifier made by hashing the emailID (creating a unique serial number by hashing the email). 
- We added a campaignID column also, which just points to which campaign a user was a part of. We added this to both trades and users dataset files
- Then we joined these two to create trades_with_trader, which includes all columns of trades + traderID. We did this by joining users and trades on the condition that the campaign ID and account ID both were same. (in a given campaign, the accountID will belong to one person only, even though across campaigns the accountID is recycled)

# Data quality checks

In [2]:
print("\nTotal trades:", len(trades_with_trader_df))
print("Distinct traders:", trades_with_trader_df["traderId"].nunique())
print("\nNulls per column:\n", trades_with_trader_df.isna().sum())
print("\nDuplicate rows:", trades_with_trader_df.duplicated().sum())



Total trades: 46376
Distinct traders: 3550

Nulls per column:
 accountId            0
closeTradeId         0
positionId           0
closeOrderId         0
openOrderId          0
durationSec          0
openDateTime         0
closeDateTime        0
profit               0
reverseProfit        0
netProfit            0
commission           0
amount               0
openPrice            0
closePrice           0
slPrice          24546
tpPrice          20886
side                 0
userGroupId          0
filename             0
campaignId           0
has_SL               0
has_TP               0
traderId             0
dtype: int64

Duplicate rows: 0


In [3]:
#Range sanity check
print(trades_with_trader_df[["durationSec", "amount", "netProfit"]].describe())

        durationSec        amount     netProfit
count  46376.000000  46376.000000  46376.000000
mean    1413.167177      0.183879     -6.662067
std     3925.868104      0.144554    106.069356
min        0.000000      0.010000   -944.100000
25%       78.000000      0.090000    -51.700000
50%      272.000000      0.120000     -0.640000
75%      995.000000      0.250000     45.400000
max    79210.000000      0.630000    680.640000


# Trader level profiling

In [4]:
trades_per_trader = trades_with_trader_df.groupby("traderId").size()
print("\nTrades per trader — mean:", trades_per_trader.mean().round(1),
      "median:", trades_per_trader.median(), "std:", trades_per_trader.std().round(1))

campaigns_per_trader = trades_with_trader_df.groupby("traderId")["campaignId"].nunique()
print("\nTraders in >1 campaign:", (campaigns_per_trader > 1).sum(), "of", len(campaigns_per_trader))

print("\nWinner/loser split:", rebuilt_account_summary["outcome"].value_counts().to_dict())



Trades per trader — mean: 13.1 median: 5.0 std: 26.7

Traders in >1 campaign: 1313 of 3550

Winner/loser split: {'loser': 2211, 'winner': 1339}


IMP : Winners = positive net profit ; Losers = negative net profit

# EDA

In [5]:
#Test 1: Median holding time, winner vs loser
hold_by_trader = trades_with_trader_df.groupby("traderId")["durationSec"].median().reset_index(name="median_hold")
rebuilt_account_summary = rebuilt_account_summary.merge(hold_by_trader, on="traderId", how="left")
w = rebuilt_account_summary[rebuilt_account_summary["outcome"]=="winner"]["median_hold"].dropna()
l = rebuilt_account_summary[rebuilt_account_summary["outcome"]=="loser"]["median_hold"].dropna()
u1, p1 = stats.mannwhitneyu(w, l)
print(f"\n[1] Holding time — winner median: {w.median():.1f}s, loser: {l.median():.1f}s, p={p1:.4f}")



[1] Holding time — winner median: 610.0s, loser: 480.0s, p=0.0009


The results are significant, winners hold longer

In [8]:
#Test 2: Commission vs directional loss

total_comm = trades_with_trader_df["commission"].sum()
total_dir = trades_with_trader_df["profit"].sum()
print(f"[2] Commission: {total_comm:.2f}, Directional: {total_dir:.2f}, Ratio: {abs(total_comm/total_dir):.1f}x")




[2] Commission: -298472.87, Directional: -10487.16, Ratio: 28.5x


In [9]:
#Test 3: No-SL 50% threshold split

nosl_pct = trades_with_trader_df.groupby("traderId")["has_SL"].apply(lambda x: 100*(~x).mean()).reset_index(name="no_sl_pct_check")
rebuilt_account_summary = rebuilt_account_summary.merge(nosl_pct, on="traderId", how="left")
high = rebuilt_account_summary[rebuilt_account_summary["no_sl_pct_check"] >= 50]["total_netProfit"]
low = rebuilt_account_summary[rebuilt_account_summary["no_sl_pct_check"] < 50]["total_netProfit"]
t3, p3 = stats.ttest_ind(high, low, equal_var=False)
print(f"[3] No-SL split — high avg profit: {high.mean():.2f}, low: {low.mean():.2f}, p={p3:.4f}")

[3] No-SL split — high avg profit: -94.78, low: -75.95, p=0.1716


The results are not significant

In [10]:
#Test 4: Position size after loss vs after win

def get_after_loss_win_sizes(df, id_col):
    df = df.sort_values([id_col, "openDateTime"]).copy()
    df["prev_netProfit"] = df.groupby(id_col)["netProfit"].shift(1)
    df["prev_outcome"] = df["prev_netProfit"].apply(lambda x: "after_loss" if x < 0 else ("after_win" if x >= 0 else None))
    sizes = df.groupby([id_col, "prev_outcome"])["amount"].mean().unstack()
    sizes.columns = ["after_loss", "after_win"]
    return sizes.dropna()

sizes = get_after_loss_win_sizes(trades_with_trader_df, "traderId")
w4, p4 = stats.wilcoxon(sizes["after_loss"], sizes["after_win"])
print(f"[4] Size after loss: {sizes['after_loss'].median():.3f}, after win: {sizes['after_win'].median():.3f}, p={p4:.4f} (n={len(sizes)})")


[4] Size after loss: 0.159, after win: 0.150, p=0.0033 (n=2043)


The results are significant but we must tread lighty here because when this test was run on the masterdataset as accountID as the unique identifier, the results were actually the opposite of this. 

In [12]:
#Test 5: Max drawdown, winner vs loser

def fast_max_drawdown(df, id_col):
    df = df.sort_values([id_col, "openDateTime"]).copy()
    df["equity"] = df.groupby(id_col)["netProfit"].cumsum()
    df["running_max"] = df.groupby(id_col)["equity"].cummax()
    df["drawdown"] = df["equity"] - df["running_max"]
    return df.groupby(id_col)["drawdown"].min().reset_index(name="max_drawdown")

dd = fast_max_drawdown(trades_with_trader_df, "traderId")
rebuilt_account_summary = rebuilt_account_summary.merge(dd, on="traderId", how="left", suffixes=("", "_dup"))
w5 = rebuilt_account_summary[rebuilt_account_summary["outcome"]=="winner"]["max_drawdown"].dropna()
l5 = rebuilt_account_summary[rebuilt_account_summary["outcome"]=="loser"]["max_drawdown"].dropna()
u5, p5 = stats.mannwhitneyu(w5, l5)
print(f"[5] Max drawdown — winner: {w5.median():.1f}, loser: {l5.median():.1f}, p={p5:.4f}")



[5] Max drawdown — winner: -30.0, loser: -230.9, p=0.0000


Strong result, this helps seperate winners from losers 

-------------------------------------------------------------------------------------------------------------

# ADDITIONAL EDA ( WE CAN ADD CODE HERE AND CLEAN UP THIS PART LATER)

------------------------------------------------------------------------------------------------------------

## EDA FROM DEVANSHI